In [1]:
library("R.matlab")
library("tidyverse")
library("afex")
library("BayesFactor")

R.matlab v3.7.0 (2022-08-25 21:52:34 UTC) successfully loaded. See ?R.matlab for help.


Attaching package: ‘R.matlab’


The following objects are masked from ‘package:base’:

    getOption, isOpen


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ purrr::%||%()   masks base::%||%()
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: lme4

Loading required package: Matrix


Attaching package: ‘Matrix’


The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack


Warning message in check_dep_version():
“ABI version mismat

In [7]:
extract_metrics <- function(filepath, group) {
  mat <- readMat(filepath)
  data.frame(
    Subject = basename(filepath),
    Group = group,
    Block = c('baseline', 'early_learning', 'late_learning'),
    ResponseTime = as.numeric(mat$meanRT[1:3]),
    MovementTime = as.numeric(mat$meanMT[1:3])
  )
}

adult_files <- list.files('adult_data', pattern = '\\_Final_Results.mat$', full.names = TRUE)
child_files <- list.files('children_data', pattern = '\\_Final_Results.mat$', full.names = TRUE)

data_adult <- map_dfr(adult_files, ~extract_metrics(.x, 'adult'))
data_child <- map_dfr(child_files, ~extract_metrics(.x, 'child'))

data_all <- bind_rows(data_adult, data_child)

head(data_all)

,Subject,Group,Block,ResponseTime,MovementTime
,<chr>,<chr>,<chr>,<dbl>,<dbl>
1,VML_MEG_011_Final_Results.mat,adult,baseline,0.3590,1.018000
2,VML_MEG_011_Final_Results.mat,adult,early_learning,0.3308,1.136467
3,VML_MEG_011_Final_Results.mat,adult,late_learning,0.3260,1.093533
4,VML_MEG_012_2_Final_Results.mat,adult,baseline,0.3590,1.018000
5,VML_MEG_012_2_Final_Results.mat,adult,early_learning,0.3308,1.136467
6,VML_MEG_012_2_Final_Results.mat,adult,late_learning,0.3260,1.093533


In [3]:
# response time anova
anova_rt <- aov_ez(
  id = "Subject",
  dv = "ResponseTime",
  data = data_all,
  between = "Group",
  within = "Block"
)

print(anova_rt)


Converting to factor: Group

Contrasts set to contr.sum for the following variables: Group



Anova Table (Type 3 tests)

Response: ResponseTime
       Effect          df  MSE       F   ges p.value
1       Group       1, 22 0.03 8.95 **  .270    .007
2       Block 1.62, 35.58 0.00    0.73  .003    .461
3 Group:Block 1.62, 35.58 0.00    0.03 <.001    .941
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘+’ 0.1 ‘ ’ 1

Sphericity correction method: GG 


In [4]:
# movement time anova
anova_mt <- aov_ez(
  id = "Subject",
  dv = "MovementTime",
  data = data_all,
  between = "Group",
  within = "Block"
)

print(anova_mt)


Converting to factor: Group

Contrasts set to contr.sum for the following variables: Group



Anova Table (Type 3 tests)

Response: MovementTime
       Effect          df  MSE         F  ges p.value
1       Group       1, 22 0.02      1.05 .033    .316
2       Block 1.33, 29.34 0.01 24.08 *** .239   <.001
3 Group:Block 1.33, 29.34 0.01      2.34 .030    .129
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘+’ 0.1 ‘ ’ 1

Sphericity correction method: GG 


In [5]:
#response time bayes factor
data_all$Subject <- as.factor(data_all$Subject)
data_all$Group <- as.factor(data_all$Group)
data_all$Block <- as.factor(data_all$Block)

bf_rt <- anovaBF(
  ResponseTime ~ Group * Block + Subject,
  data = data_all,
  whichRandom = "Subject"
)

print(bf_rt)

Bayes factor analysis
--------------
[1] Group + Subject                       : 5.124281  ±2.84%
[2] Block + Subject                       : 0.2108415 ±0.76%
[3] Group + Block + Subject               : 1.039692  ±3.73%
[4] Group + Block + Group:Block + Subject : 0.2376632 ±12.14%

Against denominator:
  ResponseTime ~ Subject 
---
Bayes factor type: BFlinearModel, JZS



In [6]:
#movement time bayes factor
bf_mt <- anovaBF(
  MovementTime ~ Group * Block + Subject,
  data = data_all,
  whichRandom = "Subject"
)

print(bf_mt)

Bayes factor analysis
--------------
[1] Group + Subject                       : 0.5101602 ±0.81%
[2] Block + Subject                       : 162656.3  ±1.8%
[3] Group + Block + Subject               : 96282.25  ±0.97%
[4] Group + Block + Group:Block + Subject : 82546.44  ±1.2%

Against denominator:
  MovementTime ~ Subject 
---
Bayes factor type: BFlinearModel, JZS

